# Using Microsoft Fabric Rest API to automate Fabric items creation.

The goal of this guide is to provide sample code for automating the creation of Microsoft Fabric items. This can be useful in various scenarios, such as CI/CD pipelines, automated environment setup for development, testing, and production, or establishing a baseline setup for organizational business units like marketing, sales, IT, and more.

**Resources**
- [Microsoft Fabric REST API documentation](https://learn.microsoft.com/en-us/rest/api/fabric/articles/)
- [Fabric API quickstart](https://learn.microsoft.com/en-us/rest/api/fabric/articles/get-started/fabric-api-quickstart)

### Prerequisites

**Permissions setup**

- You need to have Fabric capacity in order to obtain `capacity_id`. Follow this link to buy Fabric [capacity](https://learn.microsoft.com/en-us/fabric/enterprise/buy-subscription) or this to [enable Fabric for your Azure tenant](https://learn.microsoft.com/en-us/fabric/admin/fabric-switch).
- Follow [Fabric API quickstart](https://learn.microsoft.com/en-us/rest/api/fabric/articles/get-started/fabric-api-quickstart) to create app registration in Azure in order to obtain client_id, tenant_id, client_secret
- Enable **Allow service principals to use Fabric APIs** either for the entire organization or for the specific security group. Read more [here](https://learn.microsoft.com/en-us/power-bi/developer/embedded/embed-service-principal?tabs=azure-portal#step-3---enable-the-power-bi-service-admin-settings).
- In Fabric go to Settings > Admin portal > capacity settings > Fabric capacity > {click on your capacity} > contributor permissions.
  - Choose either the entire organization or specific user or groups. This to allow you to assign newly created workspaces to capacity.

**Code**
- Make sure to install dependencies > `pip install -r requirements.txt`
- We use [load_dotenv](https://pypi.org/project/python-dotenv/) to set environment variables with secrets.
- Add your values to `.env` file:

  ```
  CLIENT_ID=<replace with your client id>
  TENANT_ID=<replace with your tenant id>
  FABRIC_CLIENT_SECRET_VALUE=<replace with your secret value>
  AZURE_CAPACITY_ID=<replace with fabric capacity id>
  PRINCIPAL_ID_TO_ASSIGN=<replace with ID of your user>
  ```


In [ ]:
import requests
from azure.identity import ClientSecretCredential
from dotenv import load_dotenv
import os
import json

load_dotenv()

# Replace with your Azure AD credentials and Fabric endpoint
tenant_id = os.getenv("TENANT_ID")
client_id = os.getenv("CLIENT_ID")
client_secret = os.getenv("FABRIC_CLIENT_SECRET_VALUE")
azure_capacity_id = os.getenv("AZURE_CAPACITY_ID")
principal_id = os.getenv("PRINCIPAL_ID_TO_ASSIGN")

fabric_base_url = "https://api.fabric.microsoft.com/v1"

# Authenticate using Azure AD
def get_access_token():
    credential = ClientSecretCredential(tenant_id, client_id, client_secret)
    token = credential.get_token("https://api.fabric.microsoft.com/.default")
    return token.token

# Generate headers dynamically
def get_headers():
    token = get_access_token()
    return {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json"
    }

### List Fabric Workspaces

Let's see what workspaces you currently have. This will **not** show `MyWorkspace` that created automatically for every Fabric user.

In [ ]:
print("list_fabric_workspaces()")
def list_fabric_workspaces():
    headers = get_headers()
    url = f"{fabric_base_url}/workspaces"
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        return response.json()
    else:
        raise Exception(f"API call failed: {response.status_code}, {response.text}")
    
workspaces = list_fabric_workspaces()
for workspace in workspaces.get("value", []):
    print(f"- {workspace['displayName']} (ID: {workspace['id']}, capacityId: {workspace.get('capacityId', None)})")


### Create workspace
Now let's create a new workspace. We will call it `development`.

In [ ]:
print("create_fabric_workspace()")
def create_fabric_workspace(workspace_name, description="", capacity_id=None):
    headers = get_headers()
    url = f"{fabric_base_url}/workspaces"
    
    payload = {
        "displayName": workspace_name,
        "description": description,
        "capacityId":capacity_id
    }
    response = requests.post(url, headers=headers, json=payload)
    
    if response.status_code == 201:
        return response.json()
    else:
        raise Exception(f"Failed to create workspace: {response.status_code}, {response.text}")

workspace_name = "development"
description = "This is a test workspace created via the Fabric Rest API."
new_workspace = create_fabric_workspace(workspace_name, description, azure_capacity_id)
print("Workspace Created:")
print(f"- Name: {new_workspace['displayName']}")
print(f"- ID: {new_workspace['id']}")


### Add role assignment to a workspace

Workspace roles let you manage workspace itself and items inside.
A workspace created via the API can be managed by a Service Principal that created it. This means that no other users can view or create items within this new workspace. To grant other users permissions in the new workspace, you need to use the `roleAssignments` API.

To learn more about roles in workspaces visit [Microsoft Fabric documentation](https://learn.microsoft.com/en-us/fabric/get-started/roles-workspaces).

In the section below you will:
1. List existing workspace assignments to validate that only ServicePrincipal has Admin permissions.
2. Assign another user with Member role.

In [ ]:
print("list_workspaces_assignments()")
def list_workspaces_assignments(workspace_id):
    headers = get_headers()
    url = f"{fabric_base_url}/workspaces/{workspace_id}/roleAssignments"
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        return response.json()
    else:
        raise Exception(f"API call failed: {response.status_code}, {response.text}")
assignments = list_workspaces_assignments(new_workspace['id'])
print(f"Existing role assignments:")
print(json.dumps(assignments, indent=4))

In [ ]:
print("add_workspace_role_assignment()")
def add_workspace_role_assignment(workspace_id, principal_id):
    headers = get_headers()
    url = f"{fabric_base_url}/workspaces/{workspace_id}/roleAssignments"
    
    payload = {
        "principal": {
            "id": principal_id,
            "displayName": "MemberUser",
            "type": "User"
        },
        "role": "Member" # Admin, Member, Contributor, Viewer
    }
    response = requests.post(url, headers=headers, json=payload)
    
    if response.status_code == 201:
        return response.json()
    else:
        raise Exception(f"Failed to create lakehouse: {response.status_code}, {response.text}")
    
assigned_user = add_workspace_role_assignment(new_workspace['id'], principal_id)
print("Role assigned")

### Create lakehouse

Now that we have an active workspace we can create a Lakehouse inside it.

In [ ]:
print("create_lakehouse()")
def create_lakehouse(workspace_id, lakehouse_name, description="", creation_payload={}):
    headers = get_headers()
    url = f"{fabric_base_url}/workspaces/{workspace_id}/lakehouses"
    
    payload = {
        "displayName": lakehouse_name,
        "description": description,
        "creationPayload": creation_payload
    }
    response = requests.post(url, headers=headers, json=payload)
    
    if response.status_code == 201:
        return response.json()
    else:
        raise Exception(f"Failed to create lakehouse: {response.status_code}, {response.text}")

workspace_id=new_workspace['id']
lakehouse_name = f"{workspace_name}_lakehouse"
description = "This is a lakehouse created via Fabric Rest API.."
creation_payload = {
    "enableSchemas": True
}
new_lakehouse = create_lakehouse(workspace_id, lakehouse_name, description, creation_payload)
print("Lakehouse Created:")
print(f"- Name: {new_lakehouse['displayName']}")
print(f"- ID: {new_lakehouse['id']}")

## Cleanup

Run the code below if you want to delete resources you've created during this guide.

The code will:
- delete lakehouse
- delete workspace

In [ ]:
print("delete_lakehouse()")
def delete_lakehouse(workspace_id, lakehouse_id):
    headers = get_headers()
    url = f"{fabric_base_url}/workspaces/{workspace_id}/lakehouses/{lakehouse_id}"
    response = requests.delete(url, headers=headers)
    
    if response.status_code == 200:
        return True
    else:
        raise Exception(f"Failed to delete lakehouse: {response.status_code}, {response.text}")

delete_lakehouse(workspace_id, new_lakehouse['id'])
print("Lakehouse deleted")

In [ ]:
print("delete_workspace()")
def delete_workspace(workspace_id):
    headers = get_headers()
    url = f"{fabric_base_url}/workspaces/{workspace_id}"
    response = requests.delete(url, headers=headers)
    
    if response.status_code == 200:
        return True
    else:
        raise Exception(f"Failed to delete workspace: {response.status_code}, {response.text}")

delete_workspace(workspace_id)
print("Workspace deleted")